# Returns and value functions: how RL measures "how good"

_Reinforcement Learning_

**The return is the discounted sum of future rewards; a value function is its expectation. Bellman makes that expectation recursive.**

An RL agent collects rewards over time. To act well it needs one number per state (or per
       state-action) answering: "if I start here and keep behaving this way, how much reward will I rack up
       in the long run?" That number is a value function.

       The raw long-run reward of a single run is the return $G_t$ &mdash; the discounted sum of all
       future rewards. Because the world and the policy are random, the return is random, so we take its
       expectation. That expectation is the value function.

---

This notebook is a practice scaffold for the **Returns and value functions: how RL measures "how good"** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — numpy

We'll evaluate a fixed policy on a tiny 4-state chain MDP. We build it in three steps: (1) define the MDP's transitions and rewards, (2) collapse the dynamics under the policy we're evaluating, (3) solve the Bellman expectation equation for the value function and read off the action values.

### Step 1 — Define the chain MDP

The world is a 4-state chain: `s0 -> s1 -> s2 -> s3`, where `s3` is an absorbing goal. There are two actions: **advance** (try to step toward the goal — succeeds 80% of the time, slips and stays 20%) and **stay** (remain in place). Entering the goal pays `+1`; every other move costs `-0.1`.

`P[a]` holds the transition matrix for each action and `R[a]` the per-transition rewards. From these we compute `Rsa`, the expected immediate reward `r(s, a)` for taking action `a` in state `s`.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

# ============================================================
# A 4-state chain MDP:  s0 -> s1 -> s2 -> s3 (absorbing goal)
# Actions: 0 = advance (toward goal), 1 = stay
# ============================================================
n_states = 4
n_actions = 2
gamma = 0.9

# P[a][s, s'] = probability of next state s' given state s, action a.
P = np.zeros((n_actions, n_states, n_states))
# advance: 80% step forward, 20% slip and stay; goal absorbs.
P[0] = np.array([[0.2, 0.8, 0.0, 0.0],
                 [0.0, 0.2, 0.8, 0.0],
                 [0.0, 0.0, 0.2, 0.8],
                 [0.0, 0.0, 0.0, 1.0]])
# stay: remain in place; goal absorbs.
P[1] = np.array([[1.0, 0.0, 0.0, 0.0],
                 [0.0, 1.0, 0.0, 0.0],
                 [0.0, 0.0, 1.0, 0.0],
                 [0.0, 0.0, 0.0, 1.0]])

# R[a][s, s'] = reward for that move: +1 for entering the goal, -0.1 step cost otherwise.
R = np.full((n_actions, n_states, n_states), -0.1)
R[:, :, 3] = 1.0     # entering goal pays +1
R[:, 3, 3] = 0.0     # absorbing goal: no further reward

# Expected immediate reward r(s, a) = sum_s' P(s'|s,a) R(s,a,s').
Rsa = np.einsum('asx,asx->as', P, R)          # shape (a, s)

### Step 2 — Collapse the dynamics under the policy

A value function is tied to a specific policy. The policy we're **evaluating** picks *advance* 70% of the time and *stay* 30% in every non-goal state (and just stays once in the goal).

To evaluate it, we average the per-action transitions and rewards over the policy's action probabilities. That gives the **policy-induced** dynamics `P_pi[s, s']` and reward `R_pi[s]` — a plain Markov *reward* process with the actions integrated out.

In [ ]:
# ------------------------------------------------------------
# The fixed policy we are EVALUATING:
#   advance w.p. 0.7, stay w.p. 0.3 in every non-goal state;
#   in the goal, just stay.
# ------------------------------------------------------------
pi = np.zeros((n_states, n_actions))
pi[:, 0] = 0.7
pi[:, 1] = 0.3
pi[3] = [0.0, 1.0]

# Policy-induced dynamics:  P_pi[s, s'] and R_pi[s]
P_pi = np.einsum('sa,asx->sx', pi, P)         # average transitions over the policy
R_pi = np.einsum('sa,as->s', pi, Rsa)         # average immediate reward over the policy

### Step 3 — Solve Bellman and read off the values

For a fixed policy the Bellman *expectation* equation `V = R_pi + gamma * P_pi V` is **linear** in `V`. Rearranged, `(I - gamma P_pi) V = R_pi`, which `np.linalg.solve` cracks exactly — no iteration needed.

From `V` we recover the action values `Q(s, a) = r(s, a) + gamma * sum_s' P(s'|s,a) V(s')`. The final check confirms the identity `V(s) = sum_a pi(a|s) Q(s, a)`: a state's value is just the policy-weighted average of its action values.

In [ ]:
# ============================================================
# SOLVE THE BELLMAN EXPECTATION EQUATION DIRECTLY (linear system):
#     V = R_pi + gamma P_pi V   ==>   (I - gamma P_pi) V = R_pi
# ============================================================
V = np.linalg.solve(np.eye(n_states) - gamma * P_pi, R_pi)

# Action values from V:  Q(s,a) = r(s,a) + gamma sum_s' P(s'|s,a) V(s')
Q = Rsa.T + gamma * np.einsum('asx,x->sa', P, V)   # shape (s, a)

print("V_pi  (state values):", V)
print("Q_pi  (rows = state, cols = [advance, stay]):")
print(Q)
print("check V = sum_a pi(a|s) Q(s,a):", np.einsum('sa,sa->s', pi, Q))

# ---- output ----
# V_pi  (state values): [0.291 0.547 0.854 0.   ]
# Q_pi  (rows = state, cols = [advance, stay]):
# [[0.346 0.162]
#  [0.614 0.393]
#  [0.934 0.669]
#  [0.    0.   ]]
# check V = sum_a pi(a|s) Q(s,a): [0.291 0.547 0.854 0.   ]

## Visualize the data & results

_On a 3x3 gridworld, what is each cell worth to a random-walking agent — and do values rise toward the goal?_

### Step 1 — Build a 3x3 gridworld and its random-walk dynamics

Now a slightly bigger world to picture values spatially: a 3x3 grid where cell `8` (bottom-right) is the absorbing `+1` goal and every other step costs `-0.04`. The policy is **uniform-random** over the four moves; bumping a wall keeps you in place.

`nxt(s, a)` is the transition. We loop over states and actions to assemble the policy-induced `P_pi` and `R_pi`, exactly as before — each of the four moves contributes `0.25` of the probability and reward.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2, suppress=True)

# ----- tiny self-contained 3x3 gridworld (NO gym needed) -----
# cell index = r*3 + c.  Goal = cell 8 (bottom-right), +1, absorbing.
# Policy: uniform-random over 4 moves; bumping a wall keeps you in place.
nrows = 3
ncols = 3
N = nrows * ncols
goal = 8
gamma = 0.9
step_r = -0.04

def nxt(s, a):
    r, c = divmod(s, ncols)
    if a == 0:
        r2, c2 = r - 1, c      # up
    elif a == 1:
        r2, c2 = r + 1, c      # down
    elif a == 2:
        r2, c2 = r, c - 1      # left
    else:
        r2, c2 = r, c + 1      # right
    if 0 <= r2 < nrows and 0 <= c2 < ncols:
        return r2 * ncols + c2
    return s                            # bumped a wall -> stay

# Build policy-induced P_pi and reward R_pi for the uniform-random policy.
P_pi = np.zeros((N, N))
R_pi = np.zeros(N)
for s in range(N):
    if s == goal:
        P_pi[s, s] = 1.0                            # absorbing
        R_pi[s] = 0.0
        continue
    for a in range(4):
        sp = nxt(s, a)
        P_pi[s, sp] += 0.25                          # 1/4 per move
        R_pi[s] += 0.25 * (1.0 if sp == goal else step_r)

### Step 2 — Solve for the value of every cell

Same linear solve as the chain MDP: `(I - gamma P_pi) V = R_pi` gives the exact value of each cell under the random walk. We reshape the flat value vector back into a 3x3 grid so it lines up with the world's layout, then print it.

In [ ]:
# Solve the Bellman expectation equation directly: (I - gamma P_pi) V = R_pi
V = np.linalg.solve(np.eye(N) - gamma * P_pi, R_pi)
grid = V.reshape(nrows, ncols)
print(grid)
# [[-0.11 -0.05  0.04]
#  [-0.05  0.08  0.32]
#  [ 0.04  0.32  0.  ]]

### Step 3 — Draw the value heatmap

A heatmap makes the structure obvious: values rise smoothly toward the bottom-right goal and dip in the far corner, the farthest point from the reward. We annotate each cell with its value so the gradient toward the goal is easy to read off.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(grid, cmap="viridis")
for r in range(nrows):
    for c in range(ncols):
        ax.text(c, r, round(grid[r, c], 2), ha="center", va="center", color="white")
ax.set_xticks(range(ncols))
ax.set_yticks(range(nrows))
ax.set_title("V_pi over the gridworld (values rise toward the +1 goal)")
fig.colorbar(im, ax=ax)
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A policy gives a fixed reward stream $R_{t+1}=R_{t+2}=\cdots = 2$ forever, with $\gamma = 0.9$. What is the return $G_t$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write $G_t = 2 + 2\gamma + 2\gamma^2 + \cdots = 2\sum_{k\ge 0}\gamma^k$. — _Every reward is $2$, each discounted by a higher power of $\gamma$ — a geometric series._
- Use the geometric-series sum $\sum_{k\ge 0}\gamma^k = \frac{1}{1-\gamma}$ (valid since $\gamma\lt 1$). — _This closed form is why values stay finite even over an infinite horizon._
- Plug in: $G_t = 2\cdot\frac{1}{1-0.9} = 2\cdot 10 = 20$. — _$\frac{1}{1-\gamma}=10$ is the effective horizon at $\gamma=0.9$._

**Answer:** $G_t = \dfrac{2}{1-0.9} = 20$. Because rewards are deterministic here, the value $V^\pi$ equals this return, $20$.

</details>

**Problem 2.** In a state $s$, the policy picks action $a_1$ with probability $0.7$ and $a_2$ with probability $0.3$. You know $Q^\pi(s,a_1)=10$ and $Q^\pi(s,a_2)=4$. What is $V^\pi(s)$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the link $V^\pi(s)=\sum_a \pi(a\mid s)\,Q^\pi(s,a)$. — _A state's value is the policy-weighted average of its action values._
- Plug in: $V^\pi(s) = 0.7\times 10 + 0.3\times 4$. — _Weight each $Q$ by how often the policy picks that action._
- Compute: $7 + 1.2 = 8.2$. — _Just an expectation over the two action choices._

**Answer:** $V^\pi(s) = 0.7(10) + 0.3(4) = 8.2$. Note this is between the two $Q$-values — a value can never exceed the best action's value or fall below the worst.

</details>

**Problem 3.** Why must value functions always carry a policy superscript, $V^\pi$ rather than just $V$? Give the one-line reason and a consequence.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- State the definition: $V^\pi(s)=\mathbb{E}_\pi[G_t\mid S_t=s]$. — _The expectation is taken under policy $\pi$ — the actions are drawn from $\pi(a\mid s)$._
- Note the dynamics are fixed but the action choices are not. — _Change $\pi$ and the distribution of future rewards changes, so the average return changes._

**Answer:** Because the return averages over actions drawn from $\pi$, the value is a property of the policy, not of the state alone. Consequence: two different policies generally give two different values at the same state, and the Bellman equation you solve depends on $P_\pi$ and $R_\pi$, both built from $\pi$.

</details>